# Genotype–Phenotype Heterogeneity in NC-21OHD Dataset Exploration with `mlcroissant`This notebook guides you through loading, exploring, and analyzing the FAIR² dataset: *Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency* using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.### Dataset SourceThe dataset source is provided via a Croissant schema URL:`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}\nDescription: {metadata['description']}")

# Optional: print basic info
print(f"Published: {metadata['datePublished']}, Identifier: {metadata['identifier']}")

## 2. Data OverviewReview available record sets, fields, and their IDs.Note: All entities are referenced by their `@id` as per Croissant schema standards.

In [ ]:
# Retrieve record set IDs (using @id for all references)
record_sets = dataset.record_sets
record_set_ids = [rset['@id'] for rset in record_sets]
print("Record Sets in the Dataset:")
for rset in record_sets:
    print(f"- {rset['@id']}: {rset.get('name', '')}")
    if 'fields' in rset:
        print("  Fields:")
        for field in rset['fields']:
            print(f"    - {field['@id']} ({field.get('name', '')})")

# Preview a few records from the first record set by @id
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id:
    print(f"\nSample records from Record Set {first_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data ExtractionLoad data from each record set into a DataFrame for analysis. All record sets, fields, and columns are referenced by their `@id`.**Example:** We extract data from all record sets according to their Croissant `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set: {record_set_id}, DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("---")

# Choose a record set for further EDA (for demonstration)
target_record_set_id = record_set_ids[0] if record_set_ids else None
if target_record_set_id:
    print(f"\nRecord Set columns for EDA: {dataframes[target_record_set_id].columns.tolist()}")
    dataframes[target_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.**Croissant Note:** Always reference fields by their `@id`.

In [ ]:
# Example EDA: Filter, normalize, group

record_set_id = target_record_set_id
df = dataframes.get(record_set_id)

# Find a numeric field (@id) for demonstration
numeric_field_id = None
if df is not None:
    # Try to select a numeric field from columns
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64'] or (df[col].apply(lambda x: isinstance(x, (int, float))).mean() > 0.8):
            numeric_field_id = col
            break

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n", filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field (@id)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].nunique() > 1 and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:\n", grouped_df.head())
else:
    print("No suitable numeric field found for this record set.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.**Note:** We reference fields by their Croissant `@id`.

In [ ]:
# Visualization example: Numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and df is not None and numeric_field_id in df:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field found, show boxplot
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. ConclusionSummarize key findings and observations from the dataset exploration.- The FAIR² dataset contains clinical, laboratory, imaging, and genetic tabulations for three female NC-21OHD patients.- Record sets and fields are accessible using Croissant `@id`. Fields include numeric and categorical attributes (e.g., hormone measurements, phenotype features).- Small sample size limits broad inference, but demonstrates how Croissant schema and `mlcroissant` enable reproducible multi-modal data loading and analysis.- Further EDA and visualization provide insight into variable distributions and relationships, showing potential for clinical research pipelines.

_End of notebook. To extend the analysis, select other record sets and explore their fields using the `@id` referencing convention throughout._